In [1]:
# Imports and settings
import pandas as pd
import numpy as np

model_order = ['RSM', 'GP', 'ANN', 'Ensemble']
colors = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']

In [2]:
cols = ['DESIGN_TYPE','MODEL','RESPONSE','SCALING_METHOD','NUM_FACTORS','TRAIN_SIZE','TEST_SIZE']
met0 = pd.read_excel('Metricsv1.xlsx', usecols= cols +['MASE','MAPE','TOPOLOGY','EPOCHS','KERNEL_FUNCTION','ACTIVATION_FUNCTION'])
met1 = pd.read_excel('Metrics.xlsx')
met0.shape,met1.shape

((564, 13), (584, 13))

In [4]:
# met = pd.merge(met0, met1, on=cols, how='outer', suffixes=('_a', '_b'), indicator=True)
# met.shape,met._merge.value_counts()

In [5]:
# '''
#     # Check for comparison results
#     better: MASE_a < MASE_b and MAPE_a < MAPE_b
#     worse: MASE_a > MASE_b and MAPE_a > MAPE_b
#     equal: MASE_a == MASE_b and MAPE_a == MAPE_b
# '''
# met['MASE_COMPARISON'] = np.where(met['MASE_a'] < met['MASE_b'], 'better',
#                              np.where(met['MASE_a'] > met['MASE_b'], 'worse', 'equal'))
                             
# met['MAPE_COMPARISON'] = np.where(met['MAPE_a'] < met['MAPE_b'], 'better',
#                              np.where(met['MAPE_a'] > met['MAPE_b'], 'worse', 'equal'))

# all_cols = ['MASE_a', 'MASE_b', 'MASE_COMPARISON', 'MAPE_a', 'MAPE_b', 
#             'MAPE_COMPARISON','EPOCHS_a','EPOCHS_b','KERNEL_FUNCTION_a',
#             'KERNEL_FUNCTION_b','TOPOLOGY_a','TOPOLOGY_b','ACTIVATION_FUNCTION_a','ACTIVATION_FUNCTION_b']

# met_df = met[cols + all_cols]  
# # met_df.to_csv('model_comparison.csv', index=False)               

In [6]:
# '''
# Keep records where MASE_COMPARISON is 'better'
# '''
# met_df['MASE'] = np.where(met_df['MASE_COMPARISON'] == 'better', met_df['MASE_a'], met_df['MASE_b'])
# met_df['MAPE'] = np.where(met_df['MAPE_COMPARISON'] == 'better', met_df['MAPE_a'], met_df['MAPE_b'])
# met_df[cols].sort_values(by='RESPONSE')
# met_df.head(10)

In [ ]:
# Match by keys (including duplicate occurrences), then choose which row to keep
key_cols = ['DESIGN_TYPE', 'MODEL', 'RESPONSE']

m0 = met0.copy()
m1 = met1.copy()

# Handle duplicate key combinations by pairing each occurrence in order
m0['_pair'] = m0.groupby(key_cols).cumcount()
m1['_pair'] = m1.groupby(key_cols).cumcount()

merged = m0.merge(
    m1,
    on=key_cols + ['_pair'],
    how='outer',
    suffixes=('_0', '_1'),
    indicator=True
)

# Keep met1 when: record exists only in met1 OR met1 has smaller MASE
use_met1 = (
    merged['_merge'].eq('right_only')
    | (merged['_merge'].eq('both') & merged['MASE_1'].lt(merged['MASE_0']))
)

# Keep met0 when: record exists only in met0 OR met0 is better/equal
use_met0 = (
    merged['_merge'].eq('left_only')
    | (merged['_merge'].eq('both') & ~merged['MASE_1'].lt(merged['MASE_0']))
)

result_cols = list(met0.columns) + [c for c in met1.columns if c not in met0.columns]
met_best = pd.DataFrame(index=merged.index)

for col in result_cols:
    c0 = f'{col}_0' if f'{col}_0' in merged.columns else col
    c1 = f'{col}_1' if f'{col}_1' in merged.columns else col

    if c0 in merged.columns and c1 in merged.columns:
        met_best[col] = np.where(use_met1, merged[c1], merged[c0])
    elif c1 in merged.columns:
        met_best[col] = merged[c1]
    else:
        met_best[col] = merged[c0]

met_best = met_best.reset_index(drop=True)
print(met_best.shape)
met_best = met_best.sort_values(by=['DESIGN_TYPE','RESPONSE','MODEL'], ascending=[True, True, False]).reset_index(drop=True)
met_best.head()

,DESIGN_TYPE,SCALING_METHOD,RESPONSE,NUM_FACTORS,TRAIN_SIZE,TEST_SIZE,MODEL,MASE,MAPE,TOPOLOGY,EPOCHS,KERNEL_FUNCTION,ACTIVATION_FUNCTION
0,2LFD,CODED,inv_sqrt_avg,3.0,20.0,8.0,RSM,0.207046,3.761395,NaN,NaN,NaN,NaN
1,2LFD,STANDARDIZED,inv_sqrt_avg,3.0,20.0,8.0,GP,0.601025,10.137281,NaN,NaN,Gaussian,NaN
2,2LFD,RSM:CODED; GP/ANN:STANDARDIZED,inv_sqrt_avg,3.0,20.0,8.0,Ensemble,0.211536,3.822820,10-5,73.0,Gaussian,Tanh
3,2LFD,STANDARDIZED,inv_sqrt_avg,3.0,20.0,8.0,ANN,0.272793,5.705902,8-5,95.0,NaN,Tanh
4,2LFD,CODED,y28,2.0,13.0,1.0,RSM,0.076089,1.624837,NaN,NaN,NaN,NaN


In [10]:
met_best.to_csv('Metrics_consolidated.csv', index=False)  

In [15]:
# met[met['_merge'] == 'right_only'][cols].sort_values(by='RESPONSE')
# met[met['_merge'] == 'left_only'][cols].sort_values(by='RESPONSE')